# DLS walkthrough: reacquired 3HAE and 5HTB unannealed samples (6 November 2026)

This notebook analyzes only `20261106_3HAE_5HTB_unannealed_reacquired.exp`. It contains 2 measurement blocks and 80 acquisitions (40 per block). The sample blocks are kept separate throughout processing; acquisitions are not pooled.

The parsing, per-acquisition processing, fit-exclusion accounting, radius calculation, and plotting functions live in the adjacent `dls_walkthrough.py` module. Notebook cells import that module, call its relevant functions, and display the results.


## 1. Setup and selected data

The notebook locates the DLS folder, imports the analysis helpers, and uses only `20261106_3HAE_5HTB_unannealed_reacquired.exp`.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["figure.dpi"] = 130

working_dir = Path.cwd().resolve()
search_roots = [working_dir, *working_dir.parents]
DLS_DIR = next(
    (candidate for root in search_roots
     for candidate in (root, root / "origami_data" / "DLS")
     if (candidate / "dls_walkthrough.py").is_file()),
    None,
)
if DLS_DIR is None:
    raise FileNotFoundError("Could not locate origami_data/DLS/dls_walkthrough.py")
DATA_PATH = DLS_DIR / "data" / "20261106_3HAE_5HTB_unannealed_reacquired.exp"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Could not find the requested data file: {DATA_PATH}")
PLOT_DIR = DLS_DIR / "plots" / DATA_PATH.stem
PLOT_DIR.mkdir(parents=True, exist_ok=True)
if str(DLS_DIR) not in sys.path:
    sys.path.insert(0, str(DLS_DIR))

from dls_walkthrough import (
    analyze_hydrodynamic_radii,
    load_exp_file,
    measurement_overview,
    plot_accepted_polydispersity,
    plot_autocorrelations_by_time,
    plot_median_correlation_comparison,
    plot_autocorrelations,
    plot_hydrodynamic_radii,
    summarize_exclusions,
    summarize_hydrodynamic_radii,
)

print(f"Data file: {DATA_PATH}")
print(f"Plots will be saved to: {PLOT_DIR}")

## 2. Load and inspect the measurement blocks

`load_exp_file` reads the delay-time vector, laser wavelength, sample labels from each measurement section, acquisition elapsed times and temperatures, and every acquisition's `c1` curve. The table reports acquisition counts, temperature ranges, and elapsed-time spans before fitting begins.


In [2]:
experiment = load_exp_file(DATA_PATH)
overview = pd.DataFrame(measurement_overview(experiment))

display(overview)
print(f"Laser wavelength: {experiment.wavelength_nm:.2f} nm")
print(
    "Recorded acquisition temperatures (°C):",
    sorted({round(item.temperature_c, 2) for item in experiment.acquisitions}),
)


,measurement_id,sample,measurement,acquisitions,acquisition_duration_s,temperature_min_c,temperature_max_c,elapsed_start_s,elapsed_end_s,curve_points
0,1,Sample 1,RNA-origami_25C_5HTB_unannealed_reacquired,40,30,25.0003,25.0172,29.4,1432.6,399
1,2,Sample 2,RNA-origami_25C_3HAE_unannealed_reacquired,40,30,24.9999,25.0018,1893.4,3063.8,399


Laser wavelength: 662.83 nm
Recorded acquisition temperatures (°C): [25.0, 25.01, 25.02]


## 3. View the autocorrelation curves

Each 30-second acquisition is processed independently; curves are not averaged across samples before fitting. The instrument exports one `c1` curve per acquisition. The next cell plots the individual curves faintly and overlays each sample's pointwise median. Lag time is logarithmic so the short-time and long-time parts of the decay can both be inspected. Curves are shown as exported; baseline correction and normalization happen separately for each acquisition during fitting.

In [3]:
correlation_figure, correlation_axes = plot_autocorrelations(experiment)

for ax, row in zip(correlation_axes.flat, overview.itertuples()):
    ax.set_title(row.measurement)

correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_autocorrelations.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

### Compare median curves across samples

This overlay makes differences in the typical decay shape easier to compare between measurements. It uses each sample’s pointwise median of the exported curves.

In [4]:
median_correlation_figure, median_correlation_axis = (
    plot_median_correlation_comparison(experiment)
)
for line, row in zip(median_correlation_axis.lines, overview.itertuples()):
    line.set_label(row.measurement)
median_correlation_axis.legend(frameon=False)
median_correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_median_autocorrelation.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

### Check correlation curves over acquisition time

Curves are colored by elapsed acquisition time, with the pointwise median overlaid in black. Gradual shifts or outliers across the color scale can flag time-dependent changes or unstable acquisitions for follow-up; this view is diagnostic and does not by itself establish data quality.

In [5]:
time_correlation_figure, time_correlation_axes = (
    plot_autocorrelations_by_time(experiment)
)
for axis, row in zip(time_correlation_axes, overview.itertuples()):
    axis.set_title(row.measurement)
time_correlation_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_autocorrelations_by_time.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 4. Process each acquisition and decide whether to exclude it

Each acquisition is fitted independently using its own correlation values, lag times, and recorded temperature. The procedure is:

1. Estimate a baseline from the median of the final 30 correlation points and an amplitude from the median of the first 10 points minus that baseline.
2. Convert the normalized intensity-correlation curve to an estimated field-correlation curve using a Siegert-relation approximation, clip its lower bound for safe logarithms, and smooth it with a five-point moving window.
3. Select the decay interval from the first smoothed value below 0.8 through the first subsequent value below 0.2. Fit a quadratic to the log of the smoothed curve over that interval.
4. Derive a decay rate and PDI from the fitted coefficients. Convert the decay rate to a diffusion coefficient, then to hydrodynamic radius and diameter using the recorded temperature and assumed optical/solvent parameters. Mathematical details are in the separate appendix cells at the end.

The fit requires at least 30 paired correlation/lag points, a positive amplitude, a decay spanning both thresholds, at least eight fit points, positive lag times, and nonzero variation in the fit window. If any of these checks fails, or the estimated decay rate is not positive or the PDI is not finite, no radius is calculated. An acquisition is excluded from the final per-sample median if these requirements fail, the PDI is outside [0, 1], or the fit R-squared is below 0.80. Missing acquisition metadata or a `c1` curve raises an error rather than being silently dropped. When a fit produces a decay rate but fails a final quality gate, its radius remains visible in the acquisition table and as a cross in the plot, but it is excluded from the accepted-fit median.

The exclusion table below reports the count for each reason and how many rejected acquisitions still had a calculable radius. The full acquisition table makes each fit decision traceable to its sample and acquisition ID.


In [6]:
MIN_R_SQUARED = 0.80
SCATTERING_ANGLE_DEG = 90.0
REFRACTIVE_INDEX = 1.333
VISCOSITY_PA_S = None

fit_results = analyze_hydrodynamic_radii(
    experiment,
    min_r_squared=MIN_R_SQUARED,
    scattering_angle_deg=SCATTERING_ANGLE_DEG,
    refractive_index=REFRACTIVE_INDEX,
    viscosity_pa_s=VISCOSITY_PA_S,
)
fit_table = pd.DataFrame(fit_results)
fit_summary = pd.DataFrame(summarize_hydrodynamic_radii(fit_results))
exclusion_table = pd.DataFrame(summarize_exclusions(fit_results))

print("Per-sample accepted-size summary")
display(fit_summary)

print("Rejected acquisitions by reason")
display(exclusion_table)

Per-sample accepted-size summary


,measurement_id,sample,measurement,temperature_mean_c,acquisitions,fits_accepted,fits_rejected,median_accepted_Rh_nm,median_accepted_diameter_nm
0,1,Sample 1,RNA-origami_25C_5HTB_unannealed_reacquired,25.002417,40,13,27,148274.360539,296548.721078
1,2,Sample 2,RNA-origami_25C_3HAE_unannealed_reacquired,25.000567,40,3,37,21.103924,42.207848


Rejected acquisitions by reason


,sample,measurement,exclusion_reason,acquisitions_excluded,radius_calculated_but_excluded
0,Sample 1,RNA-origami_25C_5HTB_unannealed_reacquired,PDI outside 0-1 fit range,5,5
1,Sample 1,RNA-origami_25C_5HTB_unannealed_reacquired,fit R-squared below threshold,22,22
2,Sample 2,RNA-origami_25C_3HAE_unannealed_reacquired,PDI outside 0-1 fit range,9,9
3,Sample 2,RNA-origami_25C_3HAE_unannealed_reacquired,fit R-squared below threshold,23,23
4,Sample 2,RNA-origami_25C_3HAE_unannealed_reacquired,non-positive decay rate,5,0


## 5. Acquisition-level fit audit

The preceding table groups exclusions by sample and reason. This table preserves the fit status for every acquisition: a missing radius means a valid decay rate could not be calculated; a rejected row with a radius failed a final quality gate and is not included in the sample median.

In [7]:
display(
    fit_table.set_index(["sample", "acquisition"])[
        [
            "elapsed_s",
            "temperature_c",
            "pdi",
            "fit_r_squared",
            "hydrodynamic_radius_nm",
            "hydrodynamic_diameter_nm",
            "fit_accepted",
            "rejection_reason",
        ]
    ].round(3)
)

elapsed_s  temperature_c    pdi  fit_r_squared  \
sample   acquisition                                                   
Sample 1 1                 29.4         25.017  0.565          0.529   
         2                 59.4         25.014 -0.029          0.769   
         3                 97.5         25.011  0.491          0.893   
         4                161.1         25.007  0.412          0.904   
         5                191.1         25.006  0.590          0.693   
...                         ...            ...    ...            ...   
Sample 2 36              2943.7         25.000  0.549          0.566   
         37              2973.7         25.000  0.277          0.432   
         38              3003.8         25.000    NaN            NaN   
         39              3033.8         25.000  1.322          0.172   
         40              3063.8         25.000  0.368          0.684   

                      hydrodynamic_radius_nm  hydrodynamic_diameter_nm  \
sample   acquisition                                                     
Sample 1 1                        173126.609                346253.217   
         2                        891663.351               1783326.702   
         3                        350970.389                701940.779   
         4                           129.213                   258.426   
         5                        158518.974                317037.948   
...                                      ...                       ...   
Sample 2 36                       136891.010                273782.021   
         37                            0.565                     1.129   
         38                              NaN                       NaN   
         39                            2.127                     4.254   
         40                            0.098                     0.195   

                      fit_accepted               rejection_reason  
sample   acquisition                                               
Sample 1 1                   False  fit R-squared below threshold  
         2                   False      PDI outside 0-1 fit range  
         3                    True                           None  
         4                    True                           None  
         5                   False  fit R-squared below threshold  
...                            ...                            ...  
Sample 2 36                  False  fit R-squared below threshold  
         37                  False  fit R-squared below threshold  
         38                  False        non-positive decay rate  
         39                  False      PDI outside 0-1 fit range  
         40                  False  fit R-squared below threshold  

[80 rows x 8 columns]

## 6. Plot acquisition-level Rh and accepted-fit medians

The plot uses a logarithmic radius axis because individual estimates span a wide range. Circles are accepted fits; crosses are rejected fits that still produced a radius estimate. Acquisitions with no usable decay fit have no radius point. Each sample has a dotted horizontal line at the median hydrodynamic radius of its accepted fits, with the value printed next to the line. Rejected fits are excluded from this median.

In [8]:
radius_figure, radius_axes = plot_hydrodynamic_radii(fit_results)
radius_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_hydrodynamic_radii.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 7. Plot accepted-fit PDI and overall median PDI

Each panel shows the polydispersity index for accepted acquisitions in one sample. The dotted line and annotation show the median PDI across that sample’s accepted fits; rejected fits are excluded.

In [9]:
pdi_figure, pdi_axes = plot_accepted_polydispersity(fit_results)
pdi_figure.savefig(
    PLOT_DIR / f"{DATA_PATH.stem}_accepted_pdi.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 7. Interpretation and limitations

Use the measurement overview and fit summaries to compare samples while accounting for their recorded temperatures; samples measured at different temperatures should not be treated as a temperature-independent comparison. The estimated radii are acquisition-level, model-based values, not instrument-reported sizes. Interpret medians cautiously when few fits are accepted, and report no accepted median when a sample has zero passing fits. Confirm lag-time units, scattering geometry, solvent viscosity, and the instrument's correlation convention before drawing quantitative conclusions.


## Mathematical appendix 1: baseline, amplitude, and normalized field correlation

For one acquisition, let the exported intensity-correlation values be $c_1(\tau_i)$. The analysis estimates the long-lag baseline $b$ and initial amplitude $A$ as

$$
b = \operatorname{median}(c_1[-30:]), \qquad
A = \operatorname{median}(c_1[:10]) - b.
$$

Assuming the exported curve is a normalized intensity autocorrelation, the Siegert-relation approximation used in the code is

$$
g_1(\tau_i) = \sqrt{\operatorname{clip}\left(\frac{c_1(\tau_i)-b}{A}, 10^{-8}, \infty\right)}.
$$

The code smooths $g_1$ with a five-point moving average and selects the interval beginning at the first value below $0.8$ and ending just before the first later value below $0.2$.

## Mathematical appendix 2: cumulants fit and exclusion gates

Within the selected decay interval, define $s$ as the median lag time, $x_i=\tau_i/s$, and $y_i=\ln(g_1(\tau_i))$. The code fits

$$
y_i = a + b_1 x_i + b_2 x_i^2.
$$

The first and second cumulants are estimated from the fitted coefficients as

$$
\Gamma = -\frac{b_1}{s}, \qquad
\mathrm{PDI} = \frac{2b_2}{b_1^2}.
$$

The fit quality is computed as

$$
R^2 = 1 - \frac{\sum_i (y_i-\widehat{y}_i)^2}{\sum_i(y_i-\overline{y})^2}.
$$

The code excludes a fit from the final accepted-size summary unless $\Gamma>0$, $0\leq\mathrm{PDI}\leq 1$, and $R^2\geq 0.80$. It also rejects acquisitions where the curve, normalization, decay interval, or regression is not usable. The reported exclusion-reason table counts each acquisition once, using the first applicable final gate.

## Mathematical appendix 3: diffusion and hydrodynamic size

For scattering angle $\theta$, refractive index $n$, and laser wavelength $\lambda$, the scattering-vector magnitude is

$$
q = \frac{4\pi n}{\lambda}\sin\left(\frac{\theta}{2}\right).
$$

The translational diffusion coefficient follows from the decay rate as

$$
D = \frac{\Gamma}{q^2}.
$$

Using the Stokes--Einstein relation, the hydrodynamic radius and diameter are

$$
R_h = \frac{k_B T}{6\pi\eta D}, \qquad
d_h = 2R_h = \frac{k_B T}{3\pi\eta D}.
$$

Here $T$ is absolute temperature in kelvin, $k_B$ is the Boltzmann constant, and $\eta$ is dynamic viscosity. The notebook uses the recorded acquisition temperature, assumes 90-degree scattering and refractive index 1.333, and approximates water viscosity by interpolation between its 25 °C and 37 °C reference values. The laser wavelength comes from the `.exp` metadata.

## Mathematical appendix 4: per-sample summary

Only radii from accepted fits enter the reported sample median. For sample $j$ with accepted-acquisition set $\mathcal{A}_j$,

$$
\widetilde{R}_{h,j} = \operatorname{median}_{i\in\mathcal{A}_j}(R_{h,i}), \qquad
\widetilde{d}_{h,j} = 2\widetilde{R}_{h,j}.
$$

Rejected acquisitions are not included in these medians, even when a radius could be calculated; those diagnostic estimates remain visible in the per-acquisition table and radius plot.

In [10]:
def export_dls_plot_data(output_path=None):
    """Export all acquisition-level PDI and size estimates, including rejected fits."""
    import numpy as np
    from dls_walkthrough import (
        WATER_VISCOSITY_TEMPERATURES_C,
        WATER_VISCOSITIES_PA_S,
    )

    export_data = fit_table.copy()
    pdi = pd.to_numeric(export_data["pdi"], errors="coerce")
    radius = pd.to_numeric(export_data["hydrodynamic_radius_nm"], errors="coerce")
    accepted = export_data["fit_accepted"].astype(bool)
    finite_pdi = np.isfinite(pdi)
    valid_radius = np.isfinite(radius) & radius.gt(0)

    export_data["included_in_analysis"] = accepted
    export_data["excluded_from_summary"] = ~accepted
    export_data["pdi_outlier"] = pdi.notna() & ~pdi.between(0, 1)
    export_data["included_in_pdi_plot"] = accepted & finite_pdi
    export_data["included_in_radius_plot"] = valid_radius
    export_data["size_estimate_available"] = valid_radius
    export_data["source_file"] = str(DATA_PATH)
    export_data["wavelength_nm"] = experiment.wavelength_nm
    export_data["minimum_fit_r_squared"] = MIN_R_SQUARED
    export_data["scattering_angle_deg"] = SCATTERING_ANGLE_DEG
    export_data["refractive_index"] = REFRACTIVE_INDEX
    export_data["viscosity_pa_s"] = np.interp(
        export_data["temperature_c"],
        WATER_VISCOSITY_TEMPERATURES_C,
        WATER_VISCOSITIES_PA_S,
    )
    export_data["viscosity_model"] = "Water viscosity interpolated by acquisition temperature"

    if output_path is None:
        output_path = DATA_PATH.with_name(f"{DATA_PATH.stem}_plot_data.csv")
    output_path = Path(output_path)
    export_data.to_csv(output_path, index=False)
    print(f"Exported {len(export_data)} acquisition rows to {output_path}")
    return export_data


plot_data_export = export_dls_plot_data()

Exported 80 acquisition rows to /workspaces/codespaces-jupyter/origami_data/DLS/data/20261106_3HAE_5HTB_unannealed_reacquired_plot_data.csv
